# Predictive Performance

**Fine-tuned EmbeddingGemma 2 version** of `main/code/03_2_predictive_performance_txt.ipynb`: the DoubleMLDeep product embeddings are replaced by [EmbeddingGemma 2](https://huggingface.co/google/embeddinggemma-2) embeddings fine-tuned with LoRA and the loss of the DoubleMLDeep model on the training sample (`finetune_gemma.py`, `00_finetune_gemma_colab.ipynb`; text only, Matryoshka 256 dimensions). Datasets are read from `Gemma-tuned/data`, raw data from `main/data`, outputs are written to `Gemma-tuned/outputs`.

In [ ]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)

In [ ]:
txt_only = True
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"
df_full_train = pd.read_csv(f"../data/{dataframe_name}_train.zip")
df_full_val = pd.read_csv(f"../data/{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2",
    "P_bb_t-2",
    "REVIEW_COUNT_t-2",
    "RATING_t-2",
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val = df_full_val.dropna()

dummy_subcat_names = [
    category for category in df_full_val["subcat_aggregated"].unique()
]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

## Variable Names

In [ ]:
n_lags = 1

outcome = "Q_t"
treatment = "P_bb_t"

outcome_diff = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + [
        "Lightning Deals: Upcoming Deal",
        "Buy Box: Is FBA",
    ]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = [
    "RATING",
    "REVIEW_COUNT",
]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0",
    "similarity_cluster_1",
    "similarity_cluster_2",
    "similarity_cluster_3",
    "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls

additional_controls_deep = [
    var for var in additional_controls if var not in dummy_subcat_names
]

controls_emb = [var for var in df_full_train.columns if "emb" in var]

In [ ]:
# dataframe for all results
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)

## Performance Deep Model with time independent controls

The deep models are the DoubleMLDeep models of the main analysis (`pred_ml_*` are taken unchanged from the datasets of the main analysis). They are reported as benchmark for the predictive performance of the fine-tuned EmbeddingGemma 2 features below. The row `Fine-tuned EmbeddingGemma 2 (heads)` reports the predictions of the two heads trained together with the LoRA adapters (`finetune_gemma.py`, `data/finetuned/txt/predictions_txt.parquet`; time-independent, levels only), the counterpart of `Deep Time Independent`.

In [ ]:
df_dict = {
    "Train": df_full_train,
    "Test": df_full_val,
}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome: {r2_ml_l}")
    print(f"R2 Treatment: {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome (time dependent): {r2_ml_l_lag1}")
    print(f"R2 Treatment (time dependent): {r2_ml_m_lag1}")

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

# predictions of the heads of the fine-tuned EmbeddingGemma 2 model (time-independent, levels only)
pred_tuned = pd.read_parquet("../data/finetuned/txt/predictions_txt.parquet")
pred_tuned["date"] = pred_tuned["date"].astype(str)
for df_name, df in df_dict.items():
    p = df[["ASIN", "date"]].astype({"date": str}).merge(pred_tuned, on=["ASIN", "date"], how="left", validate="one_to_one")
    assert p[["pred_ml_l", "pred_ml_m"]].notna().all().all()
    results_df_deep.loc["Fine-tuned EmbeddingGemma 2 (heads)", f"R2 Q {df_name}"] = np.round(r2_score(df[outcome].values, p["pred_ml_l"].values), 4)
    results_df_deep.loc["Fine-tuned EmbeddingGemma 2 (heads)", f"R2 P {df_name}"] = np.round(r2_score(df[treatment].values, p["pred_ml_m"].values), 4)

results_df_deep

In [ ]:
df_dict = {
    "Train": df_full_train,
    "Test": df_full_val,
}

results_df_diff_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y_diff = df["Delta_Q_t"].values
    d_diff = df["Delta_P_bb_t"].values

    pred_ml_l_diff = df["pred_ml_l_diff"].values
    pred_ml_m_diff = df["pred_ml_m_diff"].values

    r2_ml_l_diff = np.round(r2_score(y_diff, pred_ml_l_diff), 8)
    r2_ml_m_diff = np.round(r2_score(d_diff, pred_ml_m_diff), 8)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome: {r2_ml_l_diff}")
    print(f"R2 Treatment: {r2_ml_m_diff}")

    pred_ml_l_diff_lag1 = df["pred_ml_l_diff_lag_1"].values
    pred_ml_m_diff_lag1 = df["pred_ml_m_diff_lag_1"].values

    r2_ml_l_diff_lag1 = np.round(r2_score(y_diff, pred_ml_l_diff_lag1), 8)
    r2_ml_m_diff_lag1 = np.round(r2_score(d_diff, pred_ml_m_diff_lag1), 8)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome (time dependent): {r2_ml_l_diff_lag1}")
    print(f"R2 Treatment (time dependent): {r2_ml_m_diff_lag1}")

    results_df_diff_deep[f"R2 Q {df_name}"] = (r2_ml_l_diff, r2_ml_l_diff_lag1)
    results_df_diff_deep[f"R2 P {df_name}"] = (r2_ml_m_diff, r2_ml_m_diff_lag1)

results_df_diff_deep

## Feature generation

In [ ]:
# train set
y_train = df_full_train[outcome].squeeze()
d_train = df_full_train[treatment].squeeze()
y_train_diff = df_full_train[outcome_diff].squeeze()
d_train_diff = df_full_train[treatment_diff].squeeze()

x_train = df_full_train[additional_controls]
x_train = sm.add_constant(x_train)

x_train_pca = df_full_train[additional_controls + controls_pca]
x_train_pca = sm.add_constant(x_train_pca)
x_train_sim = df_full_train[additional_controls + controls_similarities]
x_train_sim = sm.add_constant(x_train_sim)


# test set
y_test = df_full_val[outcome].squeeze()
d_test = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test = df_full_val[additional_controls]
x_test = sm.add_constant(x_test)

x_test_pca = df_full_val[additional_controls + controls_pca]
x_test_pca = sm.add_constant(x_test_pca)
x_test_sim = df_full_val[additional_controls + controls_similarities]
x_test_sim = sm.add_constant(x_test_sim)

# update names for boosting
x_train.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

x_train_pca.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test_pca.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

x_train_sim.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test_sim.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

In [ ]:
train_dict = {
    "y": y_train,
    "y_diff": y_train_diff,
    "d": d_train,
    "d_diff": d_train_diff,
    "x": x_train,
    "x_pca": x_train_pca,
    "x_sim": x_train_sim,
}

test_dict = {
    "y": y_test,
    "y_diff": y_test_diff,
    "d": d_test,
    "d_diff": d_test_diff,
    "x": x_test,
    "x_pca": x_test_pca,
    "x_sim": x_test_sim,
}

## Performance Level Data

In [ ]:
feature_specifications = ["x", "x_pca", "x_sim"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print("\n ------------------------------------")
    print(f"Feature specification: {feature_specification}")

    print("\nOLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train = r2_score(
        train_dict["y"], outcome_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Outcome (train): {ols_outcome_train}")
    ols_outcome_test = r2_score(
        test_dict["y"], outcome_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Outcome (test): {ols_outcome_test}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(
        train_dict["d"], treatment_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Treatment (train): {ols_treatment_train}")
    ols_treatment_test = r2_score(
        test_dict["d"], treatment_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Treatment (test): {ols_treatment_test}")

    print("\nBoosting")
    boost_q_outcome = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_outcome.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q_outcome.score(
        train_dict[feature_specification], train_dict["y"]
    )
    print(f"R2 Outcome (train): {r2_boost_q_train}")
    r2_boost_q_test = boost_q_outcome.score(
        test_dict[feature_specification], test_dict["y"]
    )
    print(f"R2 Outcome (test): {r2_boost_q_test}")

    boost_q_treatment = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_treatment.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_q_treatment.score(
        train_dict[feature_specification], train_dict["d"]
    )
    print(f"R2 Treatment (train): {r2_boost_d_train}")
    r2_boost_d_test = boost_q_treatment.score(
        test_dict[feature_specification], test_dict["d"]
    )
    print(f"R2 Treatment (test): {r2_boost_d_test}")

    results_df_tab_tmp = pd.DataFrame(
        data=[
            [
                ols_outcome_train,
                ols_outcome_test,
                ols_treatment_train,
                ols_treatment_test,
            ],
            [r2_boost_q_train, r2_boost_q_test, r2_boost_d_train, r2_boost_d_test],
        ],
    )
    results_df_tab = pd.concat([results_df_tab, results_df_tab_tmp], axis=0)

row_names = [
    "OLS (Tabular)",
    "Boosting (Tabular)",
    "OLS (Tabular + PCA)",
    "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)",
    "Boosting (Tabular + Similarities)",
]

results_df_tab.columns = column_names
results_df_tab.index = row_names

results_df_tab

Summary

In [ ]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

## Performance Diff Data

In [ ]:
feature_specifications = ["x", "x_pca", "x_sim"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print("\n ------------------------------------")
    print(f"Feature specification: {feature_specification}")

    print("\nOLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train = r2_score(
        train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Outcome (train): {ols_outcome_train}")
    ols_outcome_test = r2_score(
        test_dict["y_diff"], outcome_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Outcome (test): {ols_outcome_test}")

    treatment_reg = sm.OLS(
        train_dict["d_diff"], train_dict[feature_specification]
    ).fit()
    ols_treatment_train = r2_score(
        train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Treatment (train): {ols_treatment_train}")
    ols_treatment_test = r2_score(
        test_dict["d_diff"], treatment_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Treatment (test): {ols_treatment_test}")

    print("\nBoosting")
    boost_q_outcome = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_outcome.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q_outcome.score(
        train_dict[feature_specification], train_dict["y_diff"]
    )
    print(f"R2 Outcome (train): {r2_boost_q_train}")
    r2_boost_q_test = boost_q_outcome.score(
        test_dict[feature_specification], test_dict["y_diff"]
    )
    print(f"R2 Outcome (test): {r2_boost_q_test}")

    boost_q_treatment = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_treatment.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_q_treatment.score(
        train_dict[feature_specification], train_dict["d_diff"]
    )
    print(f"R2 Treatment (train): {r2_boost_d_train}")
    r2_boost_d_test = boost_q_treatment.score(
        test_dict[feature_specification], test_dict["d_diff"]
    )
    print(f"R2 Treatment (test): {r2_boost_d_test}")

    results_df_diff_tab_tmp = pd.DataFrame(
        data=[
            [
                ols_outcome_train,
                ols_outcome_test,
                ols_treatment_train,
                ols_treatment_test,
            ],
            [r2_boost_q_train, r2_boost_q_test, r2_boost_d_train, r2_boost_d_test],
        ],
    )
    results_df_diff_tab = pd.concat(
        [results_df_diff_tab, results_df_diff_tab_tmp], axis=0
    )

row_names = [
    "OLS (Tabular)",
    "Boosting (Tabular)",
    "OLS (Tabular + PCA)",
    "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)",
    "Boosting (Tabular + Similarities)",
]

results_df_diff_tab.columns = column_names
results_df_diff_tab.index = row_names

results_df_diff_tab

Summary

In [ ]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

## Summary

In [ ]:
results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100

In [ ]:
results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100